# YOLO + OCR: Extracting Vehicle Display Information from Images

A practical computer-vision workflow for detecting a relevant image region with **YOLOv8** and extracting text such as **ambient temperature** and **time** using **OpenCV + Tesseract OCR**.

### What you will learn
- Load and visualize an input image
- Run YOLOv8 object detection
- Extract a detected bounding box
- Crop and resize the detected region
- Define smaller Regions of Interest (ROIs) for text fields
- Preprocess the ROIs with grayscale conversion and thresholding
- Extract text using Tesseract OCR
- Clean the OCR output

> **Important:** The pretrained `yolov8n.pt` model is a general-purpose COCO model. It does **not** have a dedicated `instrument_cluster` class. For a production system, train/fine-tune YOLO on instrument-cluster data. This notebook demonstrates the YOLO → crop → ROI → OCR pipeline using the same approach as the training example.


## 1. Pipeline

**Input Image → YOLO Detection → Bounding Box Crop → Resize → Text ROIs → Image Preprocessing → Tesseract OCR → Cleaned Text**

The key idea is to separate the two tasks:

1. **YOLO** identifies/crops a larger relevant region.
2. **OCR** reads text from smaller regions inside that crop.

YOLO is therefore **not reading `31C` or `13:50` directly**.


## 2. Install the Python packages

Run this cell once in a new environment.

**Tesseract note:** `pytesseract` is a Python wrapper. The Tesseract OCR engine itself must also be installed on your system.


In [ ]:
%pip install -q ultralytics opencv-python matplotlib pytesseract

### Tesseract installation

Install the Tesseract OCR engine separately if it is not already installed.

- **Windows:** install Tesseract OCR and note the path to `tesseract.exe`.
- **Ubuntu/Debian:** install the `tesseract-ocr` system package.
- **macOS:** install Tesseract with your preferred package manager.

If Tesseract is already available on your PATH, the configuration cell below can remain unchanged.


In [ ]:
import os
import sys
import re
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
from ultralytics import YOLO

print("Python:", sys.version)
print("OpenCV:", cv2.__version__)

# If Tesseract is on PATH, leave this commented.
# Windows example:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("Tesseract:", pytesseract.get_tesseract_version())


## 3. Choose your input image

Place your image in the **same folder as this notebook**, or change `IMAGE_PATH` below.

For the example used during training, the image was named `Menu_1.jpg`.


In [ ]:
IMAGE_PATH = Path("Menu_1.jpg")

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Image not found: {IMAGE_PATH.resolve()}\n"
        "Place the image beside the notebook or update IMAGE_PATH."
    )

img = cv2.imread(str(IMAGE_PATH))

if img is None:
    raise RuntimeError("OpenCV could not read the image.")

print("Image loaded successfully")
print("Image shape:", img.shape)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Input Image")
plt.axis("off")
plt.show()


## 4. Load YOLOv8

We use the lightweight **YOLOv8n** model.

The first run may download `yolov8n.pt` automatically.


In [ ]:
model = YOLO("yolov8n.pt")
print("YOLOv8n model loaded.")


## 5. Run YOLO detection

The confidence threshold can be adjusted for experimentation.

The pretrained model may detect a general object rather than the instrument cluster itself. That is expected because the standard model was not trained with a dedicated instrument-cluster class.


In [ ]:
results = model.predict(
    source=str(IMAGE_PATH),
    conf=0.25,
    save=False
)

result = results[0]

print("Number of detections:", len(result.boxes))

for i in range(len(result.boxes)):
    class_id = int(result.boxes.cls[i])
    class_name = result.names[class_id]
    confidence = float(result.boxes.conf[i])

    print(
        f"Detection {i + 1}: "
        f"{class_name}, confidence={confidence:.2f}"
    )

result.show()


## 6. Inspect the bounding boxes

YOLO returns bounding boxes in the form:

`(x1, y1, x2, y2)`

where `(x1, y1)` is the top-left corner and `(x2, y2)` is the bottom-right corner.


In [ ]:
boxes = result.boxes.xyxy.cpu().numpy()

for i, box in enumerate(boxes):
    x1, y1, x2, y2 = map(int, box)

    class_id = int(result.boxes.cls[i])
    class_name = result.names[class_id]
    confidence = float(result.boxes.conf[i])

    print(
        f"{i + 1}: {class_name:15s} "
        f"confidence={confidence:.2f} "
        f"box=({x1}, {y1}, {x2}, {y2})"
    )


## 7. Crop the detected region

The first detected bounding box is used here for demonstration.

In a production application, select the detection according to the required class and confidence rather than blindly using `boxes[0]`.


In [ ]:
if len(boxes) == 0:
    raise RuntimeError(
        "No object was detected. Try a different image or lower the confidence threshold."
    )

x1, y1, x2, y2 = map(int, boxes[0])

detected_crop = img[y1:y2, x1:x2]

if detected_crop.size == 0:
    raise RuntimeError("The detected crop is empty.")

print("Detected crop shape:", detected_crop.shape)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(detected_crop, cv2.COLOR_BGR2RGB))
plt.title("YOLO Detected Region")
plt.axis("off")
plt.show()


## 8. Resize the detected region

Resizing can make small characters easier for OCR.

The aspect ratio is preserved while the height is normalized to 500 pixels.


In [ ]:
crop_rgb = cv2.cvtColor(detected_crop, cv2.COLOR_BGR2RGB)

h, w = crop_rgb.shape[:2]
target_height = 500
target_width = int(round(w * target_height / h))

resized = cv2.resize(
    crop_rgb,
    (target_width, target_height),
    interpolation=cv2.INTER_CUBIC
)

print("Resized crop shape:", resized.shape)

plt.figure(figsize=(10, 6))
plt.imshow(resized)
plt.title("Resized Detected Region")
plt.axis("off")
plt.show()


## 9. Define text ROIs

The instrument display contains multiple pieces of information. Instead of sending the whole image to OCR, we extract smaller **Regions of Interest (ROIs)**.

For the training example, the two fields were:
- Ambient temperature
- Time

The coordinates below are relative to the resized crop. **They are image-layout dependent.** If you use a different instrument cluster, adjust these coordinates to match your image.


In [ ]:
# ROI coordinates for the example instrument-display layout.
# Format: (row_start, row_end, col_start, col_end)

h, w = resized.shape[:2]

# Ambient temperature
r1_amb = h - 75
r2_amb = h - 35
c1_amb = 70
c2_amb = 120

# Time
r1_time = h - 80
r2_time = h - 30
c1_time = w - 120
c2_time = w - 50

amb_temp = resized[r1_amb:r2_amb, c1_amb:c2_amb]
time_roi = resized[r1_time:r2_time, c1_time:c2_time]

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(amb_temp)
plt.title("Ambient Temperature ROI")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(time_roi)
plt.title("Time ROI")
plt.axis("off")

plt.show()


## 10. Convert the ROIs to grayscale

OCR usually works on a single-channel image more easily than on a color image.


In [ ]:
amb_temp_gray = cv2.cvtColor(amb_temp, cv2.COLOR_RGB2GRAY)
time_gray = cv2.cvtColor(time_roi, cv2.COLOR_RGB2GRAY)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(amb_temp_gray, cmap="gray")
plt.title("Ambient Temperature - Grayscale")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(time_gray, cmap="gray")
plt.title("Time - Grayscale")
plt.axis("off")

plt.show()


## 11. Thresholding

Thresholding converts the grayscale image into a simpler black-and-white representation.

The example uses a global threshold of **200** and `THRESH_BINARY`. This value may need adjustment for different images, lighting conditions, displays, or camera exposure.


In [ ]:
_, binary_amb_temp = cv2.threshold(
    amb_temp_gray,
    200,
    255,
    cv2.THRESH_BINARY
)

_, binary_time = cv2.threshold(
    time_gray,
    200,
    255,
    cv2.THRESH_BINARY
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(binary_amb_temp, cmap="gray")
plt.title("Thresholded Temperature")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(binary_time, cmap="gray")
plt.title("Thresholded Time")
plt.axis("off")

plt.show()


## 12. Run Tesseract OCR

`--psm 7` tells Tesseract to treat the ROI as a single line of text.

The character whitelist limits recognition to characters expected in these fields.


In [ ]:
config = (
    "--psm 7 -l eng "
    "-c tessedit_char_whitelist="
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789:capm+-"
)

amb_temp_text = pytesseract.image_to_string(
    amb_temp_gray,
    config=config
)

time_text = pytesseract.image_to_string(
    time_gray,
    config=config
)

print("Raw ambient temperature:", repr(amb_temp_text))
print("Raw time:", repr(time_text))


## 13. Clean the OCR output

OCR output can contain spaces, line breaks, or unwanted characters. A small post-processing step makes the result easier to use in an application.


In [ ]:
amb_temp_clean = re.sub(
    r"[^A-Z0-9:capm+-]",
    "",
    amb_temp_text.upper()
)

time_clean = re.sub(
    r"[^A-Z0-9:capm+-]",
    "",
    time_text.upper()
)

print("Ambient Temperature:", amb_temp_clean)
print("Time:", time_clean)


## 14. Complete workflow

```text
                Input Image
                     │
                     ▼
              YOLOv8 Detection
                     │
                     ▼
              Bounding Box
                     │
                     ▼
              Region Crop
                     │
                     ▼
              Resize / Normalize
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
 Ambient Temperature ROI      Time ROI
          │                     │
          ▼                     ▼
      Grayscale              Grayscale
          │                     │
          ▼                     ▼
     Thresholding           Thresholding
          │                     │
          └──────────┬──────────┘
                     ▼
                Tesseract OCR
                     │
                     ▼
             Cleaned Text Data
```

### Key takeaway

**Detection and OCR are two different stages.**

- **YOLO:** finds a useful region.
- **OpenCV:** prepares the text ROI.
- **Tesseract:** converts pixels into characters.
- **Post-processing:** converts raw OCR output into application-ready values.


## 15. Making this production-ready

This notebook is intentionally a learning/portfolio workflow. For a real vehicle or embedded-vision application, consider:

1. Train a custom YOLO model to detect the instrument cluster or specific display regions.
2. Detect individual text fields instead of relying on fixed ROI coordinates when layouts vary.
3. Use adaptive thresholding when illumination changes.
4. Add denoising, sharpening, perspective correction, and upscaling when required.
5. Validate OCR results with format rules, for example `NN°C` for temperature or `HH:MM` for time.
6. Test across different camera angles, lighting conditions, display designs, and image resolutions.
7. Measure detection accuracy and OCR accuracy separately.


## 16. Try it yourself

### Exercise 1
Replace `Menu_1.jpg` with another image having a similar display layout and adjust the ROI coordinates.

### Exercise 2
Experiment with different threshold values such as 150, 180, 200, and 220.

### Exercise 3
Compare OCR results using:
- grayscale image
- thresholded image
- resized image

### Exercise 4
For a production-style solution, create a small custom dataset and train YOLO to detect:
- instrument cluster
- ambient temperature display
- time display
- other dashboard information fields


## References / Tools

- Python
- OpenCV
- Ultralytics YOLO
- Tesseract OCR
- Jupyter Notebook

**Portfolio note:** This notebook demonstrates an end-to-end computer-vision pipeline rather than claiming that the standard YOLOv8n model is an instrument-cluster detector.
